## Import library

In [1]:
import numpy as np
import pandas as pd
import evaluate
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)

/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Is Mac GPU (MPS) available?", torch.backends.mps.is_available())

Is Mac GPU (MPS) available? True


## Load dataset

In [3]:
train_df = pd.read_pickle("train_set.pkl")
test_df = pd.read_pickle("test_set.pkl")

train_df = train_df.rename(columns={"label": "num_label"})
test_df = test_df.rename(columns={"label": "num_label"})

print(train_df.shape)
print(test_df.shape)
print(train_df.columns)

(175, 7)
(44, 7)
Index(['signal_data', 'language', 'speaker', 'num_label', 'gender', 'c',
       'length'],
      dtype='str')


In [4]:
# Convert string into categorical data
train_df['language'] = train_df['language'].astype('category')
test_df['language']  = test_df['language'].astype('category')
label_names = train_df['language'].cat.categories.tolist()

# Create mapper
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for i, label in enumerate(label_names)}

train_df['label'] = train_df['language'].cat.codes
test_df['label'] = test_df['language'].cat.codes

In [5]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

In [6]:
train_df.head()

,signal_data,language,speaker,num_label,gender,c,length,label
176,"[0.000152587890625, 0.000823974609375, 0.00082...",SA,A,4,F,c,9501,3
165,"[-3.0517578125e-05, 0.0, 0.0, 0.0, 3.051757812...",FR,F,2,F,c,6201,0
126,"[9.1552734375e-05, 0.000152587890625, 0.000244...",FR,B,3,F,c,10001,0
103,"[0.0, 0.0, 0.0, -0.0078125, 0.0, -0.0078125, -...",SI,C,9,M,c,9001,4
70,"[-0.010894775390625, -0.018310546875, -0.00708...",IT,E,5,F,c,15000,1


## wav2vec2-xls-r-300m model loading
https://huggingface.co/facebook/wav2vec2-xls-r-300m

In [7]:
model_id = "facebook/wav2vec2-xls-r-300m"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

In [8]:
def preprocess_function(examples):
    # Extract the raw float arrays from 'signal_data'
    audio_arrays = examples["signal_data"]
    
    # Process audio arrays. XLS-R expects 16000Hz arrays
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        max_length=16000, # Adjust based on your max sample length (e.g., 16000 = 1 sec)
        truncation=True,
        padding="max_length" # Padding ensures uniform tensor shapes for stable training
    )
    
    # Pass the numerical labels through
    inputs["label"] = examples["label"]
    return inputs

In [9]:
encoded_datasets = raw_datasets.map(
    preprocess_function, 
    batched=True, 
    remove_columns=raw_datasets["train"].column_names
)

Map: 100%|██████████| 44/44 [00:00<00:00, 647.91 examples/s]


In [10]:
# Model initialization
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(label_names),
    label2id=label2id,
    id2label=id2label
)

Loading weights: 100%|██████████| 422/422 [00:00<00:00, 94850.02it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
model.freeze_feature_encoder()
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

# UPDATED: Removed use_mps_device=True
training_args = TrainingArguments(
    output_dir="./wav2vec2-low-resource-lang",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=2,   
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   
    gradient_checkpointing=True,     
    num_train_epochs=15,             
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["test"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

# --- 6. Train ---
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,12.883678,1.607063,0.250000
2,12.790063,1.596412,0.272727
3,12.715836,1.585875,0.272727
4,12.566098,1.578274,0.272727
5,12.436448,1.572456,0.272727
6,12.459130,1.567352,0.272727
7,12.301887,1.560078,0.272727
8,12.295459,1.547684,0.272727
9,12.115202,1.530308,0.272727
10,11.522993,1.506274,0.386364


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]
/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]
/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
/opt/miniconda3/envs/kcl/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]
/opt/miniconda3/envs/kcl/lib/

TrainOutput(global_step=165, training_loss=12.126831563313802, metrics={'train_runtime': 669.3846, 'train_samples_per_second': 3.922, 'train_steps_per_second': 0.246, 'total_flos': 7.955700606e+16, 'train_loss': 12.126831563313802, 'epoch': 15.0})

In [13]:
trainer.save_model("./final-wav2vec2-language-classifier")
print("Training complete! Best model successfully saved to ./final-wav2vec2-language-classifier")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.34it/s]

Training complete! Best model successfully saved to ./final-wav2vec2-language-classifier
